# Performance.py

This notebook is used to calculate the performance of a `sentence-transformers` model on a predefined set of validation examples. The validation set consists of 60 thousand enhanced and scrambled LOINC codes (enhancement means we have applied TtC synonym resolution, related-name insertion, word order permutation, and character deletion), 20 thousand each of long common name, short name, and display name. The file lives as a Data Asset in Azure Blob Storage and is read directly into local memory for computation.

Model performance is measured in several dimensions, each broken down by the number of neighbors `K` retrieved by the ANN search:

* Top-K accuracy: the percentage of the time that the correct standardized code is in the K highest scoring search results
* Mean rank: the average position (1st, 2nd, 3rd, etc.) in the list of returned neighbors (sorted by score) of the correct standardized code, when present
* Mean high cosine similarity: the average cosine similarity between the nonstandard input and the **highest** scoring search result--this result is not guaranteed to be the correct answer
* Mean right cosine similarity: the average cosine similarity between the nonstandard input and the **correct** standard code, if it appears in the top-K search results (for a particular search, if the right answer isn't found, than that search doesn't contribute to the mean calculation; only searches in which the right answer is present are used)
* Mean search time: the time it takes to retrieve the list of neighbors

Additionally, the encoding time for the model (the time it takes the model to transform an input free-text string into a vector of the embedding dimension) is computed and reported once (i.e. not stratified by K-value), since this time doesn't change as K increases.

While it is possible to run this notebook on CPU, we highly recommend using an appropriately powerful GPU (e.g. the NC24 A100 series) to speed up computation. The Approximate Nearest Neighbor search we use makes searching in memory almost instantaneous, but there is still a time-cost to encode each validation string input into a vector before semantic searching. Over 60 thousand encodings, this time adds up: GPU encoding is 10-15 times faster than CPU.


## Setup

Make sure that once the compute instance is running, you activate the kernel associated with the DIBBs Env in the upper right dropdown. Its packages are correctly optimized for this notebook and avoids some `numpy` instabilities plaguing Azure.

In [ ]:
pip install azure-keyvault-secrets azure-identity azure-ai-ml azureml-fsspec hnswlib

Now we'll do our basic, standard authentication work. We need all these variables to be able to access our container storage from a file mount. The `DATASTORE_NAME` is not a protected secret and therefore doesn't need to be stashed in KeyVault, as it's a standard Azure default.

In [ ]:
# Authenticate to Key Vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

_IMPORTANT_: This step can't be skipped, even though we're not directly using any of the `ml_client` functionality. This authentication and connection step allows us to use this notebook cleanly within our compute ecosystem. Basically, instantiating the class object acts as a connection that allows us to do everything that follows.

In [ ]:
from azure.ai.ml import MLClient

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

Finally, we'll set the remainder of our imports and some global constants we'll be using.

The most important variables in this cell are the `MODEL VARIABLES` values, the `MODLE_NAME` and `EMBEDDING_SIZE`. The model name comes directly from the hugging face page for a particular model, and can be directly copied using the button beside the name on the web page. The embedding size for most models we work with is 768 (which is the industry standard dimensionality for any BERT-based transformer), but some are 1024 and a few are even higher. You can refer to the spreadsheet that tracks model performance for the exact size of a given model. If you set an embedding size here that proves to be incorrect later, when calculating the index, no harm will be done--the cell will simply halt and tell you the embedding dimensionality is wrong. Just come back here and change the value (likely to whichever of 1024 or 768 you haven't tried yet), then go back and pick up with indexing.

In [ ]:
import os
import random
import time
from typing import List

# MODEL VARIABLES
MODEL_NAME = "intfloat_e5-large-v2_0.3_1e05_tuned_10000_1e05"
EMBEDDING_SIZE = 1024

# The name of the file in blob storage of the model's pickled vectors
SNOINC_CODES_FILE = "./loinc_lab_names_20251008.csv"
DATE = SNOINC_CODES_FILE.split("_")[-1].split(".")[0]

# NOTE: MAKE SURE THIS DIRECTORY IS CORRECTLY SET
# It's currently loading from a fine-tuned model, but if you're testing
# e.g. TSDAE or a non-adapted model, adjust directories accordingly
EMBEDDING_FILE = f"embeddings/refined/fine_tuned/loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"

# These variables are NOT necessary for regular use--this is purely for
# JSONL validation. Just leave this first boolean set to False unless
# you want to verify the embedding integrity of JSONL (which is a 
# very slooooow operation).
USE_JSONL_EMBEDDINGS = False
MODEL_SPECIFIC_JSONL_DIR = f"loinc_lab_names_{MODEL_NAME.replace('/', '_')}_{DATE}"
PATH_TO_JSONL_DIR = f"embeddings/refined/split/{MODEL_SPECIFIC_JSONL_DIR}"

# The name of the HNSW index file for this particular model
INDEX_FP = f"hnswlib_index_{MODEL_NAME.replace('/', '_')}.index"

# VALIDATION VARIABLES
VALIDATION_FILE = "./validation_set_60k_pairs.txt"

**Important**: This cell determines whether the notebook will use Exact Neareset Neighbor search or Approximate Nearest Neighbor Search. Exact search is supported _only_ when the notebook is run using a GPU-enabled cluster, since the Tensor operations are valid only in a CUDA environment with `pytorch`. Further, exact search is computationally feasible only with a GPU providing a massive speed boost.

For TtC team purposes, we have found ANN using a compute instance with a lot of RAM and a large number of cores to be the most performant evaluation option. ANN is roughly twice as fast as Exact search, even with GPU-boosting, due to the speed of retrieval. GPU-boosted exact search is in turn ~10 times faster than exact search without GPU-boosting. For most use-cases, we advise using ANN.

In [ ]:
import torch

USE_EXACT_SEARCH = False

if USE_EXACT_SEARCH:
    assert torch.cuda.is_available()

## Step 1: Create File Mount

The Azure Machine Learning File Mount system, though cumbersomely named, allows us to _directly_ access files and objects we have stored in the DIBBs TTC container. Any `.txt` or `.csv` files need to be created as Data Assets (see sidebar on left), while embedding tensor files and any HNSW `.index` files do not, and can simply be loaded directly from storage.

In [ ]:
# Load up the validation set data
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)


## Step 2: Unpickle Embeddings

Using our mounted file system, we can directly open the embedding file and unpickle it. Remember, each embedding file is stored as a dictionary of not just the embeddings computed by the `sentence-transformers` model, but the standard LOINC codes associated with those embeddings. These are important later for measuring accuracy.

In [ ]:
import pickle
import json
import numpy as np
import os

if not USE_JSONL_EMBEDDINGS:
    # Load up the pre-computed embeddings from the datastore
    with fs.open(EMBEDDING_FILE) as fp:
        cache_data = pickle.load(fp)

    name_codes = cache_data["codes"]
    embeddings = cache_data["embeddings"]

    if not USE_EXACT_SEARCH:
        # We move the embedded vectors to CPU for optimized searching later,
        # since the ANN indexer exists in CPU
        embeddings = embeddings.cpu().numpy()

else:
    # Construct a list of all JSONL files we'll need to open
    EMBEDDING_FILE_LIST = [
        os.path.join(PATH_TO_JSONL_DIR, os.path.basename(p))
        for p in fs.ls(PATH_TO_JSONL_DIR)
        if p.endswith(".jsonl")
    ]

    cache_data = []
    for file in EMBEDDING_FILE_LIST:
        with fs.open(file) as f:
            cache_data.extend(
                json.loads(line)
                for line in f
                if line.strip()
            )

    name_codes = [item["description"] for item in cache_data]
    embeddings = [np.array(item["descriptionVector"], dtype=np.float32)
                for item in cache_data]
    loinc_type = [item["type"] for item in cache_data]

    if not USE_EXACT_SEARCH:
        # embeddings: list of 1D vectors -> stack to 2D array for hnswlib
        embeddings = np.stack(embeddings, axis=0)  # shape (N, D)


## Step 3: Load HNSW Index

Whether computed from a previous Azure run, or computed locally and uploaded, we will use the embeddings to populate an HNSW index for fast Approximate Nearest Neighbor searching. The parameter values below govern the depth / connectivity of the search, but note that if the index was previously constructed, only the `EF_SEARCH` value will impact performance.

The `hnswlib` package _cannot_ directly open Azure binary files, which is how the FileSystemMount accesses and passes them. So what we need to do instead is first copy the file from the remote mount to local, working memory, and then we can access and open it. Once we've done that, it should be locally persisted for the remainder of our session.

During operation, this cell will create a temporary copy of the `.index` file in local, working memory. At the end of the cell, the file will be remote copied to Blob Storage and then deleted from local memory (on subsequent runs, it will simply be fetched from remote storage). You can verify the file has been cleaned by checking the sidebar to the left, under `Notebooks`.


In [ ]:
import hnswlib

# MODEL DIRECTORY
# IMPORTANT: Make sure this sub-folder is set to the correct location
# where the model's respective index file lives (or should live). For
# fine-tuned models, this should be "fine_tuned/". For TSDAE models that
# haven't been tuned, this should be "tsdae/". For all other untrained
# models, it should be "".
MODEL_SUB_DIR = "fine_tuned/"

# ANN INDEX VARIABLES
EF_CONSTRUCTION = 400
M_VALUE = 64
EF_SEARCH = 400

if not USE_EXACT_SEARCH:
    # Load up or create an index over the embedding data
    index = hnswlib.Index(space="cosine", dim=EMBEDDING_SIZE)

    # Azure will check blob storage first using the file mount
    print("Checking for cached ANN index...")
    if fs.exists("indexes/" + MODEL_SUB_DIR + INDEX_FP):
        print("  Found cached index. Loading it...")

        # First, try to regularly load the index, in case we copied it here
        # from a previous run
        try:
            index.load_index(INDEX_FP)
        
        # If we can't open the file (because it's AzureML binary), then we
        # can create a local ported copy and open from that
        except:
            try:
                fs.get("indexes/" + MODEL_SUB_DIR + INDEX_FP, '.')
                index.load_index(INDEX_FP)
            
            # If that doesn't work then the file is beyond the reach of mortal
            # hands and is best left undisturbed, like all sleeping gods
            except:
                print("Could not copy or load index")
        
    else:
        print("No locally cached index found. Creating hierarchical index...")
        index.init_index(
            max_elements=len(embeddings), ef_construction=EF_CONSTRUCTION, M=M_VALUE
        )
        index.add_items(embeddings, list(range(len(embeddings))))

        # Default is to save to local, working memory, so we'll need to remote copy
        # to Azure blob storage just like the reverse of copying from blob storage
        # Also clean up the local copy to avoid surplus memory charges
        index.save_index(INDEX_FP)
        fs.put(INDEX_FP, "/indexes/" + MODEL_SUB_DIR)
    os.remove(INDEX_FP)

    # The index should be holding approximately 276k embeddings so it better exceed 0
    assert index.get_current_count() > 0
    index.set_ef(EF_SEARCH)


## Step 4: Load Validation Set

With our file system mount, loading the validation set and preparing it for evaluation is straightforward. No need to worry about local copying for this data, Azure's `fs.open()` can simply parse the binary into a string codec for us.

In [ ]:
print("Loading validation set...")
examples = []
with fs.open(VALIDATION_FILE) as fp:
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            examples.append(line_str.strip().split("|"))

# There are either 60k examples or 240k examples in this list, depending
# on whether the abridged set or full validation set is used
assert len(examples) >= 60000

## Step 5: Perform Evaluation

This cell carries out the trial run with the model and scores its performance on the validation data. It's largely a dictionary-based tracking function that accumulates some numbers into lists partitioned out by the K-value associated with the run. The search method of retrieving results is slightly different depending on whether exact search or ANN is used (i.e. slightly different unpacking of the `hits` list). 

When we use the `hnswlib` API to perform ANN, we get a pretty nested structure of a pair of lists denoting the search results and the _distances_ of those results to the input query. The only nuance to this function is unpacking those lists, converting distances into scores (since we want to measure similarity), and pairing up the found neighbor result with the standard LOINC code it represents, using our earlier unpickled Corpus ID indices.

First, we'll appropriately load up the model.

In [ ]:
from sentence_transformers import SentenceTransformer

FETCH_TRAINED_MODEL = True
TRAINED_DIR = "fine_tuned/"

# First, check if the model exists locally--if it does, nothing to do here
if os.path.exists(MODEL_NAME):
    print("Model exists locally, loading it...")

# If there isn't a local copy, we'll fetch it from remote
else:   
    if fs.exists("models/" + TRAINED_DIR + MODEL_NAME):
        print("Found trained model, loading from remote...")
        fs.get("models/" + TRAINED_DIR + MODEL_NAME, '.')
        print("Model loaded to local memory.")
    else:
        print("Could not find model at specified path.")
        print(
            "Check model name (esp. parameter numbers, underscores, and dashes) and fetch directory."
        )
    
print("Instantiating language model...")
model = SentenceTransformer(MODEL_NAME)

In [ ]:
from sentence_transformers import CrossEncoder
from sentence_transformers import util

# IMPORTANT: The default set of K-Values should be [1,3,5,10].
# HOWEVER, if you're testing the reranker, then this should be a list
# of only a single element: K_VALUES = [10].
K_VALUES = [1, 3, 5, 10]

# Set to True if you want to see stats computed for K-Values. For reranker
# operations, this should be set to False.
SHOW_MULTI_K_STATS = True

USE_RERANKER = False
RERANKER_NAME = "mixedbread-ai/mxbai-rerank-xsmall-v1"

if USE_RERANKER:
    print("Instantiating re-ranker")
    reranker = CrossEncoder(RERANKER_NAME)

    # Stats for Re-Ranker Testing
    reranker_accuracy = 0.0
    reranker_high_sims = []
    reranker_right_sims = []

print("Predicting and computing stats for validation set...")

# Stats for Multiple K Value Testing
encoding_times = []
highest_cosine_sims = {k: [] for k in K_VALUES}
right_cosine_sims = {k: [] for k in K_VALUES}
times = {k: [] for k in K_VALUES}
ranks = {k: [] for k in K_VALUES}
examples_with_correct_output_in_top_k = {k: 0.0 for k in K_VALUES}

# Always randomize before evaluation to avoid cold-initialization bias
random.shuffle(examples)

for i, e in enumerate(examples):

    # We're impatient people, this helps keep us sane by seeing progress
    # is happening
    if i % 10_000 == 0:
      print(f"Calculated {i} of {len(examples)} examples.")

    correct_code = e[0].strip()
    nonstandard_in = e[1].strip()

    # Depending on whether we're using ANN, the encoded non-standard input
    # must be kept in CPU (ANN) or GPU (exact search)
    start = time.time()
    if USE_EXACT_SEARCH:
        enc = model.encode(nonstandard_in, convert_to_tensor=True)
    else:
        enc = model.encode(nonstandard_in)
    encoding_times.append(time.time() - start)

    for k in K_VALUES:
        start = time.time()
        if USE_EXACT_SEARCH:
            hits = util.semantic_search(enc, embeddings, top_k=k)
            hits = hits[0]
        else:
            start = time.time()
            # Slightly more complicated unpacking of approximate search results
            # Note that ANN works using _distances, so we have to convert
            # to scores (cosine distance is unit-normalized to always be 
            # length-1, so we can just subtract 1 - dist)
            embedding_ids, distances = index.knn_query(enc, k=k)
            hits = [
                {"corpus_id": id, "score": 1 - dist}
                for id, dist in zip(embedding_ids[0], distances[0])
            ]
            hits = sorted(hits, key=lambda x: x["score"], reverse=True)

        times[k].append(time.time() - start)
        highest_cosine_sims[k].append(hits[0]["score"])

        if USE_RERANKER:
            # We need to reconstruct each code possibility before feeding it
            # into the reranker
            query_hits = []
            for h in hits:
                mapped_sentence = name_codes[h["corpus_id"]]  # ty: ignore
                query_hits.append(mapped_sentence)
            
            # Now we can cross-encode to get the scores for all pairs of codes
            ranks = reranker.rank(nonstandard_in, query_hits)
            hits_idx = ranks[0]['corpus_id']
            predicted_code = query_hits[hits_idx]

            # Save the stats output
            if predicted_code == correct_code:
                reranker_accuracy += 1.0
                reranker_right_sims.append(ranks[0]['score'])
            reranker_high_sims.append(ranks[0]['score'])

        else:
            # Check if correct answer is in the returned search results
            correct_in_top_k = False
            for idx, h in enumerate(hits):
                mapped_sentence = name_codes[h["corpus_id"]]  # ty: ignore
                if mapped_sentence == correct_code:
                    correct_in_top_k = True
                    # Hits is a 0-indexed list, so translate the index to the nth
                    # element of the list
                    ranks[k].append(idx+1)
                    right_cosine_sims[k].append(hits[idx]["score"])
                    break
            if correct_in_top_k:
                examples_with_correct_output_in_top_k[k] += 1.0

if SHOW_MULTI_K_STATS:
    mean_encoding_time = round(float(sum(encoding_times)) / float(len(encoding_times)), 3)
    print(f"  Mean Encoding Time: {mean_encoding_time} seconds")

    for k in K_VALUES:
        mean_high_cosine_sim = round(float(sum(highest_cosine_sims[k])) / float(len(highest_cosine_sims[k])), 3)
        mean_right_cosine_sim = round(float(sum(right_cosine_sims[k])) / float(len(right_cosine_sims[k])), 3)
        mean_encoding_search_time = round(float(sum(times[k])) / float(len(times[k])), 3)
        top_k_accuracy = round(examples_with_correct_output_in_top_k[k] / float(len(examples)), 5)
        mean_rank = round(float(sum(ranks[k])) / float(len(ranks[k])), 3)

        print(f"  Trial: Value for Top-K at K = {k}")
        print(f"    Top-K Accuracy: {top_k_accuracy * 100.0}%")
        print(f"    Mean Rank of Correct Code (when present): {mean_rank}")
        print(f"    Mean Highest Cosine Similarity: {mean_high_cosine_sim}")
        print(f"    Mean Correct Cosine Similarity: {mean_right_cosine_sim}")
        print(f"    Mean Search Time: {mean_encoding_search_time}")

elif USE_RERANKER:
    reranker_accuracy = round(reranker_accuracy / float(len(examples)), 5)
    if len(reranker_high_sims) > 0:
        mean_reranker_high_sim = round(float(sum(reranker_high_sims)) / float(len(reranker_high_sims)), 3)
    else:
        mean_reranker_high_sim = "n/a"
    if len(reranker_right_sims) > 0:
        mean_reranker_right_sim = round(float(sum(reranker_right_sims)) / float(len(reranker_right_sims)), 3)
    else:
        mean_reranker_right_sim = "n/a"
    print(f"Re-Ranker Accuracy: {reranker_accuracy * 100.0}%")
    print(f"Mean Highest Cosine Similarity: {mean_reranker_high_sim}")
    print(f"Mean Correct Cosine Similarity: {mean_reranker_right_sim}")
